# Beyond Audio — Final downstream experiments (Stage 1)

**Reconstructed from the team's original Google Colab development notebook, November 2026.**

This **separate** notebook organizes the final **logistic-regression and fusion** experiments. It does **not** overwrite the original development notebook, train the original audio/image feature extractors, or claim to reproduce the research results until it has been executed using the verified MSD-I feature files.

**What this stage does:**
1. Validate the released-feature subset and artist-disjoint split metadata.
2. Train seven logistic-regression configurations, tuning regularization on **validation** macro-F1.
3. Tune the CNN fixed and confidence-adaptive late-fusion rules on **validation** macro-F1.
4. Evaluate selected classifiers **once on the held-out test split** and save transparent metrics/predictions.

**Not in Stage 1:** neural-network experiments, visual-typicality centroids/quintiles, bootstrap confidence intervals, perceptual-hash robustness and figures. These remain in the original Colab development notebook; a later stage can incorporate them.

**Requirements:** The 12 precomputed `.npy` matrices and three label CSVs listed below must already be present. This notebook does **not** generate those from the original MSD-I archive. See `DATA_AND_REPRODUCIBILITY.md` in the repository. None of the dataset or original pretrained weights are included in the public notebook.

**Research integrity:** Split membership and data alignment must be confirmed from the original archive before trusting results. The validation set is used to select parameters. The test set is for evaluation only. This new clean layout and code organization are subsequent work, not a record of code that existed at the start of the project.

## 1. Environment and input paths

Run this cell in Colab (or set `MSDI_PROJECT_DIR` if running elsewhere). **Do not publicly upload any credentials or private data.**

In [15]:
from pathlib import Path
import os
import sys
import random
import platform
import numpy as np
import pandas as pd
import sklearn
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Change this environment variable if your project files are in another location.
DEFAULT_ROOT = "/content/drive/MyDrive/MSD-I_project"
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
PROJECT = Path(os.environ.get("MSDI_PROJECT_DIR", DEFAULT_ROOT))
CNN_DIR = PROJECT / "finetuning_experiments" / "multimodal_artist_disjoint"
MM_DIR = PROJECT / "finetuning_experiments" / "oramas_representation_experiment"

# Completely separate output directory; existing experiment artifacts remain unchanged.
OUTPUT = PROJECT / "finetuning_experiments" / "reproducible_stage1_2026"
print("Project root:", PROJECT)
print("CNN inputs:", CNN_DIR)
print("MM inputs:", MM_DIR)
print("Python:", platform.python_version())
print("numpy:", np.__version__, "pandas:", pd.__version__, "scikit-learn:", sklearn.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/MSD-I_project
CNN inputs: /content/drive/MyDrive/MSD-I_project/finetuning_experiments/multimodal_artist_disjoint
MM inputs: /content/drive/MyDrive/MSD-I_project/finetuning_experiments/oramas_representation_experiment
Python: 3.13.18
numpy: 2.4.6 pandas: 2.2.3 scikit-learn: 1.6.1


## 2. Verify required files and metadata

Original development notebook inputs are the `.npy` arrays and `labels_{train,val,test}.csv`. The MM and CNN label rows must be identical and in the same order. **No values are filled in or invented if files are missing.**

In [16]:
SPLITS = ("train", "val", "test")
DIMS = {"cnn_audio": 2048, "cnn_visual": 2048, "mm_audio": 200, "mm_visual": 200}
SIZES = {"train": 8242, "val": 1836, "test": 1792}
REQUIRED_META = {"msd_track_id", "msd_artist_id", "genre", "split"}

missing = []
for split in SPLITS:
    missing.extend([p for p in [
        CNN_DIR / f"labels_{split}.csv",
        CNN_DIR / f"X_{split}_audio.npy",
        CNN_DIR / f"X_{split}_visual.npy",
        MM_DIR / f"labels_{split}.csv",
        MM_DIR / f"X_{split}_mm-audio.npy",
        MM_DIR / f"X_{split}_mm-visual.npy",
    ] if not p.is_file()])
if missing:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(map(str, missing)))

labels = {}
for split in SPLITS:
    cnn_labels = pd.read_csv(CNN_DIR / f"labels_{split}.csv")
    mm_labels = pd.read_csv(MM_DIR / f"labels_{split}.csv")
    for name, df in [("CNN", cnn_labels), ("MM", mm_labels)]:
        assert REQUIRED_META.issubset(df.columns), f"{name} {split}: missing label fields"
        assert df[list(REQUIRED_META)].notna().all().all(), f"{name} {split}: null metadata"
        assert df["msd_track_id"].is_unique, f"{name} {split}: duplicate track IDs"
        assert (df["split"] == split).all(), f"{name} {split}: incorrect split names"
    assert len(cnn_labels) == len(mm_labels) == SIZES[split], split
    for col in ("msd_track_id", "msd_artist_id", "genre", "split"):
        assert cnn_labels[col].astype(str).equals(mm_labels[col].astype(str)), (split, col)
    labels[split] = cnn_labels.reset_index(drop=True)

for i, a in enumerate(SPLITS):
    for b in SPLITS[i+1:]:
        assert set(labels[a].msd_track_id).isdisjoint(labels[b].msd_track_id)
        assert set(labels[a].msd_artist_id).isdisjoint(labels[b].msd_artist_id)
assert sum(map(len, labels.values())) == 11870
assert sum(df.msd_artist_id.nunique() for df in labels.values()) == 6461
assert labels["test"].msd_artist_id.nunique() == 975
classes = np.sort(labels["train"].genre.astype(str).unique())
assert len(classes) == 6
for split in SPLITS:
    assert set(labels[split].genre.astype(str)) <= set(classes)
print("Verified artist-disjoint split sizes:", {s: len(labels[s]) for s in SPLITS})
print("Classes:", list(classes))

Verified artist-disjoint split sizes: {'train': 8242, 'val': 1836, 'test': 1792}
Classes: ['Electronic', 'Jazz', 'Metal', 'Pop', 'Rap', 'Rock']


## 3. Load and validate embeddings

The visual, audio, MM, and CNN files must be aligned to the **same** track order verified above.

In [17]:
from collections import OrderedDict
features = {}
for split in SPLITS:
    arrays = OrderedDict([
        ("cnn_audio", np.load(CNN_DIR / f"X_{split}_audio.npy", allow_pickle=False)),
        ("cnn_visual", np.load(CNN_DIR / f"X_{split}_visual.npy", allow_pickle=False)),
        ("mm_audio", np.load(MM_DIR / f"X_{split}_mm-audio.npy", allow_pickle=False)),
        ("mm_visual", np.load(MM_DIR / f"X_{split}_mm-visual.npy", allow_pickle=False)),
    ])
    for name, matrix in arrays.items():
        assert matrix.shape == (SIZES[split], DIMS[name]), (split, name, matrix.shape)
        assert np.isfinite(matrix).all(), f"Non-finite features: {split} {name}"
    features[split] = arrays

Y = {s: labels[s].genre.astype(str).to_numpy() for s in SPLITS}

def concat_input(split, names):
    assert split in SPLITS
    return np.concatenate([features[split][name] for name in names], axis=1)

print("All four feature matrices verified on train / validation / test.")

All four feature matrices verified on train / validation / test.


## 4. Select logistic-regression hyperparameters using validation data only

Based on the original project: training-only `StandardScaler`, `LogisticRegression(solver="lbfgs", max_iter=2000)`, and `C` grid `{0.001, 0.01, 0.1, 1, 10}`.

**Important note about ties:** Original CNN baseline cells kept the first C when macro-F1 tied; the later MM representation experiment used sorting by macro-F1, then by smaller C. Both select the smallest tied C over this ascending grid.

Running this cell can take time on the 4,496-dimensional representations. Do not rerun it just to generate figures; saved checkpoints may be reused after they are verified.

In [ ]:
EXPERIMENTS = OrderedDict([
    ("CNN Audio LR", ("cnn_audio",)),
    ("CNN Visual LR", ("cnn_visual",)),
    ("CNN Early Fusion LR", ("cnn_audio", "cnn_visual")),
    ("MM Audio LR", ("mm_audio",)),
    ("MM Visual LR [validation support]", ("mm_visual",)),
    ("MM Early Fusion LR", ("mm_audio", "mm_visual")),
    ("All Four LR", ("cnn_audio", "cnn_visual", "mm_audio", "mm_visual")),
])
C_VALUES = [0.001, 0.01, 0.1, 1, 10]
models = {}
model_validation_rows = []

for name, feature_names in EXPERIMENTS.items():
    X_train = concat_input("train", feature_names)
    X_val = concat_input("val", feature_names)
    best_f1 = -float("inf")
    best_model, best_C = None, None
    for C in C_VALUES:
        clf = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(C=C, solver="lbfgs", max_iter=2000, random_state=SEED)),
        ])
        clf.fit(X_train, Y["train"])
        pred_val = clf.predict(X_val)
        macro_f1 = f1_score(Y["val"], pred_val, average="macro", zero_division=0)
        model_validation_rows.append({"model": name, "C": C, "validation_macro_f1": macro_f1})
        if macro_f1 > best_f1:
            best_f1, best_model, best_C = macro_f1, clf, C
    models[name] = best_model
    print(f"{name:34s} C={best_C:<6} validation macro-F1={best_f1:.4f}")

validation_grid = pd.DataFrame(model_validation_rows)
assert len(models) == len(EXPERIMENTS)
assert all(np.array_equal(model.classes_, classes) for model in models.values())

## 5. Tune fixed and adaptive late fusion **on validation**

These are CNN late-fusion methods, matching the report. Fixed-late fusion tests 21 weights; adaptive fusion tests 12 pairs `(beta, gamma)`.

In [19]:
pa = models["CNN Audio LR"].predict_proba(concat_input("val", EXPERIMENTS["CNN Audio LR"]))
pv = models["CNN Visual LR"].predict_proba(concat_input("val", EXPERIMENTS["CNN Visual LR"]))
assert pa.shape == pv.shape == (SIZES["val"], len(classes))
assert np.array_equal(models["CNN Audio LR"].classes_, models["CNN Visual LR"].classes_)

def evaluate_fusion_probabilities(proba, true_y):
    predictions = classes[np.argmax(proba, axis=1)]
    return f1_score(true_y, predictions, average="macro", zero_division=0)

late_grid, adaptive_grid = [], []
fixed_best = (-np.inf, None)
for alpha in np.linspace(0.0, 1.0, 21):
    score = evaluate_fusion_probabilities(alpha * pa + (1-alpha) * pv, Y["val"])
    late_grid.append({"alpha_audio": float(alpha), "validation_macro_f1": score})
    if score > fixed_best[0]:
        fixed_best = (score, float(alpha))

adaptive_best = (-np.inf, None, None)
confidence = pv.max(axis=1)
for beta in [0.25, 0.5, 0.75, 1.0]:
    for gamma in [1, 2, 3]:
        weights = beta * confidence ** gamma
        fused = (1 - weights[:, None]) * pa + weights[:, None] * pv
        score = evaluate_fusion_probabilities(fused, Y["val"])
        adaptive_grid.append({"beta": beta, "gamma": gamma, "validation_macro_f1": score})
        if score > adaptive_best[0]:
            adaptive_best = (score, beta, gamma)

alpha = fixed_best[1]
beta, gamma = adaptive_best[1:]
print("Selected fixed audio weight:", alpha)
print("Selected adaptive beta / gamma:", beta, gamma)
assert alpha is not None and beta is not None

Selected fixed audio weight: 0.55
Selected adaptive beta / gamma: 0.5 1


## 6. Final held-out test evaluation

This cell performs **no tuning on test labels**. It saves results under a new output directory. The MM visual-only classifier was trained for validation support, but is **not included** in the report's principal test performance table.

In [20]:
TEST_EXPERIMENTS = [
    "CNN Audio LR", "CNN Visual LR", "CNN Early Fusion LR",
    "MM Audio LR", "MM Early Fusion LR", "All Four LR"
]
rows = []
prediction_table = labels["test"].copy()

def add_metrics(name, pred):
    return dict(model=name,
                accuracy=100 * accuracy_score(Y["test"], pred),
                balanced_accuracy=100 * balanced_accuracy_score(Y["test"], pred),
                macro_f1=100 * f1_score(Y["test"], pred, average="macro", zero_division=0))

for name in TEST_EXPERIMENTS:
    pred = models[name].predict(concat_input("test", EXPERIMENTS[name]))
    prediction_table[name] = pred
    rows.append(add_metrics(name, pred))

pa_test = models["CNN Audio LR"].predict_proba(concat_input("test", EXPERIMENTS["CNN Audio LR"]))
pv_test = models["CNN Visual LR"].predict_proba(concat_input("test", EXPERIMENTS["CNN Visual LR"]))
fixed_test = classes[np.argmax(alpha * pa_test + (1-alpha) * pv_test, axis=1)]
visual_weights_test = beta * pv_test.max(axis=1) ** gamma
adaptive_test_prob = (1-visual_weights_test[:, None]) * pa_test + visual_weights_test[:, None] * pv_test
adaptive_test = classes[np.argmax(adaptive_test_prob, axis=1)]
for name, pred in [("CNN Fixed Late Fusion", fixed_test), ("CNN Adaptive Late Fusion", adaptive_test)]:
    prediction_table[name] = pred
    rows.append(add_metrics(name, pred))

metrics_table = pd.DataFrame(rows)
print(metrics_table.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

OUTPUT.mkdir(parents=True, exist_ok=True)
metrics_table.to_csv(OUTPUT / "stage1_test_metrics.csv", index=False)
prediction_table.to_csv(OUTPUT / "stage1_test_predictions.csv", index=False)
validation_grid.to_csv(OUTPUT / "stage1_validation_lr_grid.csv", index=False)
pd.DataFrame(late_grid).to_csv(OUTPUT / "stage1_validation_late_grid.csv", index=False)
pd.DataFrame(adaptive_grid).to_csv(OUTPUT / "stage1_validation_adaptive_grid.csv", index=False)
for name, model in models.items():
    safe_name = name.lower().replace(" ", "_").replace("[", "").replace("]", "")
    joblib.dump(model, OUTPUT / f"stage1_{safe_name}.joblib")
print("Saved output to:", OUTPUT)

                   model  accuracy  balanced_accuracy  macro_f1
            CNN Audio LR     65.90              62.36     64.07
           CNN Visual LR     47.60              44.76     46.34
     CNN Early Fusion LR     69.64              67.39     68.68
             MM Audio LR     66.52              63.13     64.64
      MM Early Fusion LR     70.20              67.30     69.10
             All Four LR     69.48              67.51     68.69
   CNN Fixed Late Fusion     68.69              65.02     67.29
CNN Adaptive Late Fusion     68.30              64.97     67.08
Saved output to: /content/drive/MyDrive/MSD-I_project/finetuning_experiments/reproducible_stage1_2026


## 7. Check against the paper without silently altering reported results

These values are **reference values stated in the submitted report**, not results generated by this notebook. Treat discrepancies as a reason to investigate data files, software versions, original tie-breaking or code changes. **Do not overwrite the reported findings automatically.**

Neural-network results, visual-typicality tables and bootstrap intervals are **not** reproduced by Stage 1.

In [21]:
REPORT_REFERENCE = {
    "CNN Audio LR": (65.90, 62.36, 64.07),
    "CNN Visual LR": (47.60, 44.76, 46.34),
    "CNN Early Fusion LR": (69.64, 67.39, 68.68),
    "CNN Fixed Late Fusion": (68.69, 65.02, 67.29),
    "CNN Adaptive Late Fusion": (68.30, 64.97, 67.08),
    "MM Audio LR": (66.52, 63.13, 64.64),
    "MM Early Fusion LR": (70.20, 67.30, 69.10),
    "All Four LR": (69.48, 67.51, 68.69),
}
comparison_rows = []
for _, rec in metrics_table.iterrows():
    expected = REPORT_REFERENCE.get(rec["model"])
    if expected is None:
        continue
    comparison_rows.append({
        "model": rec["model"],
        "max_abs_difference_pp": max(abs(float(rec[k]) - e) for k, e in zip(
            ("accuracy", "balanced_accuracy", "macro_f1"), expected
        )),
    })
comparison = pd.DataFrame(comparison_rows)
print(comparison.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("These are descriptive checks; the original results must be independently verified.")

                   model  max_abs_difference_pp
            CNN Audio LR                  0.004
           CNN Visual LR                  0.004
     CNN Early Fusion LR                  0.005
             MM Audio LR                  0.004
      MM Early Fusion LR                  0.004
             All Four LR                  0.005
   CNN Fixed Late Fusion                  0.004
CNN Adaptive Late Fusion                  0.004
These are descriptive checks; the original results must be independently verified.


In [22]:

# Compare original saved predictions with Stage 1 predictions.
# No model training or test-set tuning takes place here.

original_results = {
    "CNN Early Fusion LR": (
        CNN_DIR / "final_test_predictions.csv",
        "early_fusion_prediction"
    ),
    "All Four LR": (
        MM_DIR / "representation_final_test_predictions.csv",
        "all_four_prediction"
    ),
}

for model_name, (original_file, original_column) in original_results.items():

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    original = pd.read_csv(
        original_file,
        dtype={"msd_track_id": str}
    )[["msd_track_id", "genre", original_column]]

    reproduced = prediction_table[
        ["msd_track_id", "genre", model_name]
    ].copy()

    reproduced["msd_track_id"] = (
        reproduced["msd_track_id"].astype(str)
    )

    comparison = reproduced.merge(
        original,
        on="msd_track_id",
        suffixes=("_new", "_original"),
        validate="one_to_one",
        how="outer",
        indicator=True
    )

    assert len(comparison) == 1792
    assert (comparison["_merge"] == "both").all()
    assert (
        comparison["genre_new"]
        == comparison["genre_original"]
    ).all()

    different = comparison[
        comparison[model_name] != comparison[original_column]
    ].copy()

    print("Number of differing predictions:", len(different))

    if len(different):
        print(different[[
            "msd_track_id",
            "genre_new",
            original_column,
            model_name
        ]].to_string(index=False))



CNN Early Fusion LR
Number of differing predictions: 0

All Four LR
Number of differing predictions: 0


In [23]:

# Diagnostic only: does not retrain or overwrite models.

checks = [
    (
        "CNN Early Fusion LR",
        CNN_DIR / "fusion_logistic_best.joblib",
        CNN_DIR / "final_test_predictions.csv",
        "early_fusion_prediction",
        "TRSPGXY128F92FFEA3"
    ),
    (
        "All Four LR",
        MM_DIR / "all_four_best.joblib",
        MM_DIR / "representation_final_test_predictions.csv",
        "all_four_prediction",
        "TRRRYFA128F93165B5"
    )
]

test_ids = labels["test"]["msd_track_id"].astype(str)

for name, model_path, csv_path, column, track_id in checks:
    print("\n" + "=" * 55)
    print(name)
    print("=" * 55)

    old_model = joblib.load(model_path)
    new_model = models[name]
    X = concat_input("test", EXPERIMENTS[name])

    original_csv = pd.read_csv(
        csv_path, dtype={"msd_track_id": str}
    ).set_index("msd_track_id")[column].reindex(test_ids)

    assert original_csv.notna().all()

    old_predictions = old_model.predict(X)
    new_predictions = new_model.predict(X)

    print(
        "Original checkpoint vs original CSV disagreements:",
        np.sum(old_predictions != original_csv.to_numpy())
    )

    print(
        "Original checkpoint vs new model disagreements:",
        np.sum(old_predictions != new_predictions)
    )

    print(
        "Original C:",
        old_model.named_steps["classifier"].C
    )
    print(
        "New C:",
        new_model.named_steps["classifier"].C
    )

    scale_difference = np.max(np.abs(
        old_model.named_steps["scaler"].mean_
        - new_model.named_steps["scaler"].mean_
    ))

    coefficient_difference = np.max(np.abs(
        old_model.named_steps["classifier"].coef_
        - new_model.named_steps["classifier"].coef_
    ))

    print("Max scaler-mean difference:", scale_difference)
    print("Max coefficient difference:", coefficient_difference)

    i = np.flatnonzero(test_ids.to_numpy() == track_id)[0]

    for label, model in [
        ("Original checkpoint", old_model),
        ("New model", new_model)
    ]:
        probabilities = model.predict_proba(X[i:i+1])[0]
        order = np.argsort(probabilities)[::-1][:2]

        print(f"\n{label}:")
        for k in order:
            print(
                f"  {model.classes_[k]}: "
                f"{probabilities[k]:.6f}"
            )



CNN Early Fusion LR
Original checkpoint vs original CSV disagreements: 0
Original checkpoint vs new model disagreements: 0
Original C: 0.001
New C: 0.001
Max scaler-mean difference: 0.0
Max coefficient difference: 0.0

Original checkpoint:
  Electronic: 0.483599
  Rock: 0.482605

New model:
  Electronic: 0.483599
  Rock: 0.482605

All Four LR
Original checkpoint vs original CSV disagreements: 0
Original checkpoint vs new model disagreements: 0
Original C: 0.001
New C: 0.001
Max scaler-mean difference: 0.0
Max coefficient difference: 0.0

Original checkpoint:
  Rock: 0.493364
  Pop: 0.493046

New model:
  Rock: 0.493364
  Pop: 0.493046


In [24]:

# Compare optimization settings without retraining.

for name, model_path, *_ in checks:
    original = joblib.load(model_path)
    reproduced = models[name]

    old = original.named_steps["classifier"]
    new = reproduced.named_steps["classifier"]

    print("\n" + "=" * 55)
    print(name)
    print("=" * 55)

    parameters = [
        "solver",
        "C",
        "tol",
        "max_iter",
        "fit_intercept",
        "class_weight",
        "random_state",
        "multi_class"
    ]

    for parameter in parameters:
        old_value = old.get_params().get(parameter)
        new_value = new.get_params().get(parameter)

        status = "MATCH" if old_value == new_value else "DIFFERENT"

        print(
            f"{parameter:16s} "
            f"Original={str(old_value):12s} "
            f"New={str(new_value):12s} "
            f"{status}"
        )

    print("Original iterations:", old.n_iter_)
    print("New iterations:", new.n_iter_)



CNN Early Fusion LR
solver           Original=lbfgs        New=lbfgs        MATCH
C                Original=0.001        New=0.001        MATCH
tol              Original=0.0001       New=0.0001       MATCH
max_iter         Original=2000         New=2000         MATCH
fit_intercept    Original=True         New=True         MATCH
class_weight     Original=None         New=None         MATCH
random_state     Original=None         New=42           DIFFERENT
multi_class      Original=deprecated   New=deprecated   MATCH
Original iterations: [130]
New iterations: [130]

All Four LR
solver           Original=lbfgs        New=lbfgs        MATCH
C                Original=0.001        New=0.001        MATCH
tol              Original=0.0001       New=0.0001       MATCH
max_iter         Original=2000         New=2000         MATCH
fit_intercept    Original=True         New=True         MATCH
class_weight     Original=None         New=None         MATCH
random_state     Original=42           New=42